[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vittot/ISCB-NLP-Course-2026/blob/main/notebooks/ex3_explainability.ipynb)

# EX 3 Explainability: SHAP vs. Aug-Linear

**Goal.** EX 1-2 built several classifiers; the most reliable one, by every metric
we tracked, was **cardiac vs. not** (best real-label F1, best-calibrated weak label,
most training data). Here we ask *why* it predicts what it predicts, comparing two very
different explainability strategies on that same task and the same fine-tuned
Bio_ClinicalBERT model:

- **SHAP on BERT**: a *post-hoc* explanation. The model stays a black box; SHAP estimates,
  for one prediction at a time, how much each input token pushed the output up or down,
  by comparing the model's output across many partial maskings of the input.
- **Aug-Linear**: an *inherently* interpretable model. Instead of raw word counts
  (TF-IDF) or a fine-tuned black box (BERT), it represents each note as a sum of
  n-gram **BERT embeddings** and fits a single linear layer on top. Because the model
  is linear in a sum of per-n-gram vectors, every n-gram gets one fixed, global
  coefficient -- "how much does this n-gram push toward cardiac" -- valid for *every*
  document it appears in, not just one. That's the trade its name describes: augment a
  transparent linear model with an LLM's semantic representations, instead of asking a
  black box for an explanation after the fact.

Both approaches end up answering a similar question -- which words and phrases drive the
prediction -- from opposite directions: SHAP explains individual predictions of a complex
model and we aggregate afterwards; Aug-Linear is a single global explanation by
construction. Comparing their top unigrams, bigrams, and trigrams tells us how much they
agree.

## 1. Setup

In [1]:
import re
import random
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, average_precision_score)

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 0

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BERT_MODEL = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LENGTH = 192
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

C:\Users\vitto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## 2. Load data and pick the task

Cardiac vs. not, using the same weak label (from block 2's clustering) and real label
(the independent rule + manual annotation) as blocks 2-3.

In [2]:
df = pd.read_csv("data/dyspnea_weak_and_real_labels.csv", dtype={"document_id": str})
texts = df["clinical_note"].values
weak_y = df["weak_cardiac"].values
real_y = df["cardiac"].values
print(df.shape, "| weak positives:", weak_y.sum(), "| real positives:", real_y.sum())

def undersample_indices(labels, neg_ratio=4, random_state=0):
    rng = np.random.RandomState(random_state)
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]
    n_neg = min(len(neg_idx), len(pos_idx) * neg_ratio)
    sampled_neg = rng.choice(neg_idx, size=n_neg, replace=False)
    return np.concatenate([pos_idx, sampled_neg])

def pick_threshold(y_val, proba_val):
    candidates = np.linspace(0.05, 0.95, 19)
    scores = [f1_score(y_val, (proba_val >= t).astype(int), zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(scores))])

# One 80/20 train/test split (not the full nested CV from blocks 2-3) -- this notebook is
# about explaining *one* canonical model, not re-running the evaluation machinery. Within
# the training portion, carve out a small validation slice (never undersampled, never
# used for fitting) to pick each model's decision threshold without leakage into the test
# fold -- same fix blocks 2-3 needed once the training data gets undersampled.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(skf.split(texts, weak_y))

inner_skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
fit_rel, val_rel = next(inner_skf.split(train_idx, weak_y[train_idx]))
fit_idx, val_idx = train_idx[fit_rel], train_idx[val_rel]

train_sub = undersample_indices(weak_y[fit_idx], neg_ratio=4, random_state=RANDOM_STATE)
fit_idx_sub = fit_idx[train_sub]
print("fit set:", len(fit_idx_sub), "rows,", weak_y[fit_idx_sub].sum(), "positive |",
      "val set:", len(val_idx), "rows |", "test set:", len(test_idx), "rows")

(2667, 9) | weak positives: 225 | real positives: 847
fit set: 675 rows, 135 positive | val set: 534 rows | test set: 534 rows


## 3. Train one canonical BERT classifier

Full fine-tune, same recipe as block 3 section 5 -- this is the model both explainability
methods will be compared against (SHAP explains it directly; Aug-Linear is a separate,
inherently-interpretable model trained on the same data, to see how much accuracy an
interpretable alternative gives up).

In [3]:
class NotesDataset(Dataset):
    def __init__(self, texts, labels, max_length=MAX_LENGTH):
        self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=max_length)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

BERT_CACHE_PATH = Path("data/block4_bert_cardiac_model.pt")
BATCH_SIZE = 8
EPOCHS = 2

def fine_tune(train_texts, train_labels, seed=RANDOM_STATE):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2).to(DEVICE)
    train_ds = NotesDataset(train_texts, train_labels)
    generator = torch.Generator(); generator.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    model.train()
    for epoch in range(EPOCHS):
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward(); optimizer.step(); optimizer.zero_grad()
    return model

def bert_predict_proba(model, texts, batch_size=BATCH_SIZE):
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = list(texts[i:i + batch_size])
            enc = tokenizer(batch_texts, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
            logits = model(**enc).logits
            probs.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

if BERT_CACHE_PATH.exists():
    print(f"Loading cached model weights from {BERT_CACHE_PATH}")
    bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2).to(DEVICE)
    bert_model.load_state_dict(torch.load(BERT_CACHE_PATH, map_location=DEVICE))
else:
    print("Fine-tuning...")
    bert_model = fine_tune(texts[fit_idx_sub], weak_y[fit_idx_sub])
    torch.save(bert_model.state_dict(), BERT_CACHE_PATH)
    print(f"Cached model weights to {BERT_CACHE_PATH}")

test_proba = bert_predict_proba(bert_model, texts[test_idx])
print("held-out ROC-AUC vs real:", round(roc_auc_score(real_y[test_idx], test_proba), 3))
print("held-out PR-AUC  vs real:", round(average_precision_score(real_y[test_idx], test_proba), 3))

Loading cached model weights from data\block4_bert_cardiac_model.pt



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights:   1%|          | 1/199 [00:00<?, ?it/s, Materializing param=bert.embeddings.LayerNorm.bias]


Loading weights:   1%|          | 1/199 [00:00<00:01, 164.22it/s, Materializing param=bert.embeddings.LayerNorm.bias]


Loading weights:   1%|          | 2/199 [00:00<00:00, 257.39it/s, Materializing param=bert.embeddings.LayerNorm.weight]

...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 676.97it/s, Materializing param=bert.pooler.dense.weight]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 676.97it/s, Materializing param=bert.pooler.dense.weight]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 663.05it/s, Materializing param=bert.pooler.dense.weight]


BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

held-out ROC-AUC vs real: 0.859
held-out PR-AUC  vs real: 0.776


## 4. SHAP: explaining individual predictions

We run SHAP's `Partition` explainer (an efficient approximation for text: it recursively
splits the input and only tests coalitions consistent with that hierarchy, instead of
every possible token subset) on a sample of notes the model calls cardiac. This gives a
Shapley value per *token* per document -- how much that token pushed this document's
prediction toward "cardiac". We then aggregate across the sample to get global unigram
importance, and approximate bigram/trigram importance as the summed SHAP value of
consecutive tokens (documented as an approximation: SHAP itself only ever scores single
mask units, so a "bigram score" here means "these two adjacent tokens' individual
contributions, added together and then aggregated across occurrences" -- not a joint
Shapley value for the pair).

In [4]:
import shap

N_EXPLAIN = 20
explain_idx = np.where(real_y[test_idx] == 1)[0][:N_EXPLAIN]
explain_texts = [texts[test_idx][i][:600] for i in explain_idx]  # cap length to bound SHAP cost
print(f"Explaining {len(explain_texts)} real-cardiac test notes (truncated to 600 chars each)")

def f(batch_texts):
    return bert_predict_proba(bert_model, list(batch_texts))

# A word-level masker (split on whitespace/punctuation), not the model's own
# WordPiece tokenizer -- BERT splits "cardiology" into "card"+"##iology", and
# masking/reporting at that granularity would give unreadable sub-word fragments
# instead of the whole words we actually want to rank.
masker = shap.maskers.Text(r"\W+")
explainer = shap.Explainer(f, masker)
shap_values = explainer(explain_texts, max_evals=300)

Explaining 20 real-cardiac test notes (truncated to 600 chars each)



PartitionExplainer explainer:  15%|█▌        | 3/20 [00:00<?, ?it/s]


PartitionExplainer explainer:  25%|██▌       | 5/20 [00:14<00:21,  1.44s/it]


PartitionExplainer explainer:  30%|███       | 6/20 [00:17<00:27,  1.96s/it]


PartitionExplainer explainer:  35%|███▌      | 7/20 [00:19<00:25,  1.97s/it]

...


PartitionExplainer explainer:  90%|█████████ | 18/20 [00:43<00:04,  2.07s/it]


PartitionExplainer explainer:  95%|█████████▌| 19/20 [00:45<00:02,  2.09s/it]


PartitionExplainer explainer: 100%|██████████| 20/20 [00:47<00:00,  2.06s/it]


PartitionExplainer explainer: 21it [00:50,  2.20s/it]                        


PartitionExplainer explainer: 21it [00:50,  2.80s/it]

In [5]:
unigram_scores, bigram_scores, trigram_scores = {}, {}, {}

def add_score(d, key, value):
    if key not in d:
        d[key] = []
    d[key].append(value)

for doc_tokens, doc_values in zip(shap_values.data, shap_values.values):
    words = [t.strip() for t in doc_tokens]
    vals = list(doc_values)
    for i, w in enumerate(words):
        if w and w.isalpha():
            add_score(unigram_scores, w.lower(), vals[i])
    for i in range(len(words) - 1):
        w1, w2 = words[i].strip(), words[i + 1].strip()
        if w1 and w2:
            add_score(bigram_scores, f"{w1.lower()} {w2.lower()}", vals[i] + vals[i + 1])
    for i in range(len(words) - 2):
        w1, w2, w3 = words[i].strip(), words[i + 1].strip(), words[i + 2].strip()
        if w1 and w2 and w3:
            add_score(trigram_scores, f"{w1.lower()} {w2.lower()} {w3.lower()}", vals[i] + vals[i + 1] + vals[i + 2])

def top_ngrams(score_dict, min_occurrences=2, top_n=15):
    rows = [(k, np.mean(v), len(v)) for k, v in score_dict.items() if len(v) >= min_occurrences]
    rows.sort(key=lambda r: -r[1])
    return pd.DataFrame(rows[:top_n], columns=["ngram", "mean_shap_value", "n_occurrences"])

shap_unigrams = top_ngrams(unigram_scores)
shap_bigrams = top_ngrams(bigram_scores)
shap_trigrams = top_ngrams(trigram_scores, min_occurrences=2)
print("=== Top unigrams (SHAP) ===\n", shap_unigrams.to_string(index=False))
print("\n=== Top bigrams (SHAP) ===\n", shap_bigrams.to_string(index=False))
print("\n=== Top trigrams (SHAP) ===\n", shap_trigrams.to_string(index=False))

=== Top unigrams (SHAP) ===
        ngram  mean_shap_value  n_occurrences
          ef         0.048782              5
  cardiology         0.031338              5
         des         0.028275              2
          af         0.023297              2
         lad         0.022034              3
      apical         0.020555              2
    anterior         0.020179              2
         tte         0.019973              2
       covid         0.017885              2
       heart         0.017618              7
    previous         0.014948              2
       chest         0.014212             10
       visit         0.014165              4
hypertension         0.013613              2
hypertensive         0.013439              2

=== Top bigrams (SHAP) ===
              ngram  mean_shap_value  n_occurrences
        chest pain         0.039625              2
hypertensive heart         0.036965              2
           mid lad         0.033605              3
             af on

## 5. Aug-Linear: a linear model over BERT-embedded n-grams

Build a vocabulary of unigrams/bigrams/trigrams, get each one's BERT embedding (mean-pooled
last hidden state, computed once, with no gradient), represent each document as the sum of
its n-grams' embeddings, and fit logistic regression on top. The result is linear in these
embeddings, so every n-gram's coefficient is just its embedding dotted with the fitted
weight vector -- one fixed number per n-gram, valid everywhere it appears.

In [6]:
vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 3), min_df=5, max_features=4000)
doc_term = vectorizer.fit_transform(texts)
vocab = vectorizer.get_feature_names_out()
print(f"vocabulary: {len(vocab)} n-grams (unigrams+bigrams+trigrams, min_df=5)")

@torch.no_grad()
def embed_texts(text_list, batch_size=64):
    embeddings = []
    bert_model.eval()
    base_model = bert_model.bert  # the encoder, without the classification head
    for i in range(0, len(text_list), batch_size):
        batch = list(text_list[i:i + batch_size])
        enc = tokenizer(batch, truncation=True, padding=True, max_length=16, return_tensors="pt").to(DEVICE)
        out = base_model(**enc).last_hidden_state  # (batch, seq, hidden)
        mask = enc["attention_mask"].unsqueeze(-1)
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        embeddings.append(pooled.cpu().numpy())
    return np.concatenate(embeddings)

NGRAM_EMB_CACHE = Path("data/block4_ngram_embeddings.pkl")
if NGRAM_EMB_CACHE.exists():
    print(f"Loading cached n-gram embeddings from {NGRAM_EMB_CACHE}")
    with open(NGRAM_EMB_CACHE, "rb") as f:
        ngram_embeddings = pickle.load(f)
else:
    print("Embedding vocabulary with BERT (no gradient, one forward pass per n-gram)...")
    ngram_embeddings = embed_texts(list(vocab))
    with open(NGRAM_EMB_CACHE, "wb") as f:
        pickle.dump(ngram_embeddings, f)
    print(f"Cached to {NGRAM_EMB_CACHE}")

print("n-gram embedding matrix:", ngram_embeddings.shape)

vocabulary: 4000 n-grams (unigrams+bigrams+trigrams, min_df=5)
Loading cached n-gram embeddings from data\block4_ngram_embeddings.pkl
n-gram embedding matrix: (4000, 768)


In [7]:
# document representation: sum of the embeddings of the n-grams it contains, weighted
# by how many times each n-gram appears (doc_term is a sparse count matrix).
doc_embeddings_raw = doc_term @ ngram_embeddings
print("document embedding matrix:", doc_embeddings_raw.shape)

def fit_and_eval(doc_embeddings, label):
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    clf.fit(doc_embeddings[fit_idx_sub], weak_y[fit_idx_sub])
    test_proba = clf.predict_proba(doc_embeddings[test_idx])[:, 1]
    roc = roc_auc_score(real_y[test_idx], test_proba)
    pr = average_precision_score(real_y[test_idx], test_proba)
    print(f"{label:35s} ROC-AUC vs real: {roc:.3f} | PR-AUC vs real: {pr:.3f}")
    return clf, test_proba

# Quick test: does L2-normalizing each document's embedding (so document length/verbosity
# stops inflating the vector's magnitude, the same way TfidfVectorizer's default
# norm="l2" already controls for length) close some of the gap to TF-IDF?
from sklearn.preprocessing import normalize
doc_embeddings_l2 = normalize(doc_embeddings_raw, norm="l2")

# And: does averaging instead of summing (so a document that happens to match more
# n-grams doesn't automatically get a larger-magnitude vector) help on its own?
ngram_counts_per_doc = np.asarray(doc_term.sum(axis=1)).ravel()
doc_embeddings_mean = doc_embeddings_raw / np.clip(ngram_counts_per_doc, 1, None)[:, None]

print("=== raw sum vs. L2-normalized vs. mean-pooled document embeddings ===")
_, _ = fit_and_eval(doc_embeddings_raw, "raw sum (original)")
auglin_clf, auglin_test_proba = fit_and_eval(doc_embeddings_l2, "L2-normalized")
_, _ = fit_and_eval(doc_embeddings_mean, "mean-pooled")

# Adopt the L2-normalized version as "Aug-Linear" for the rest of this notebook --
# see section 8 for the comparison and why.
doc_embeddings = doc_embeddings_l2

# per-n-gram score: its embedding dotted with the fitted linear weight vector -- one
# fixed number per n-gram, not tied to any single document.
ngram_scores = ngram_embeddings @ auglin_clf.coef_[0]
ngram_lengths = np.array([len(g.split()) for g in vocab])

def top_auglin_ngrams(n_words, top_n=15):
    mask = ngram_lengths == n_words
    idx = np.where(mask)[0]
    order = idx[np.argsort(-ngram_scores[idx])][:top_n]
    return pd.DataFrame({"ngram": vocab[order], "auglin_score": ngram_scores[order]})

auglin_unigrams = top_auglin_ngrams(1)
auglin_bigrams = top_auglin_ngrams(2)
auglin_trigrams = top_auglin_ngrams(3)
print()
print("=== Top unigrams (Aug-Linear, L2-normalized) ===")
print(auglin_unigrams.to_string(index=False))
print()
print("=== Top bigrams (Aug-Linear, L2-normalized) ===")
print(auglin_bigrams.to_string(index=False))
print()
print("=== Top trigrams (Aug-Linear, L2-normalized) ===")
print(auglin_trigrams.to_string(index=False))

document embedding matrix: (2667, 768)
=== raw sum vs. L2-normalized vs. mean-pooled document embeddings ===


raw sum (original)                  ROC-AUC vs real: 0.644 | PR-AUC vs real: 0.537
L2-normalized                       ROC-AUC vs real: 0.822 | PR-AUC vs real: 0.717
mean-pooled                         ROC-AUC vs real: 0.788 | PR-AUC vs real: 0.681

=== Top unigrams (Aug-Linear, L2-normalized) ===
           ngram  auglin_score
      myocardial     58.468284
    cardiologist     56.257440
      arrhythmic     53.897815
  echocardiogram     51.089669
       diastolic     49.908441
  cardiomyopathy     49.133687
     clopidogrel     48.752019
      cardiology     48.668326
echocardiography     47.615695
     ventricular     46.546915
  cardiovascular     46.542566
       cardioasa     45.717298
        systolic     43.939173
antihypertensive     43.166754
    atorvastatin     42.155738

=== Top bigrams (Aug-Linear, L2-normalized) ===
                     ngram  auglin_score
         echocardiogram ef     59.753644
          cardiology visit     58.429056
hypokinetic cardiomyopathy     58

C:\Users\vitto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 6. SHAP vs. Aug-Linear: how much do they agree?

In [8]:
def overlap(top_a, top_b, col_a="ngram", col_b="ngram"):
    set_a, set_b = set(top_a[col_a]), set(top_b[col_b])
    return len(set_a & set_b), sorted(set_a & set_b)

for n_words, shap_top, auglin_top in [
    ("unigrams", shap_unigrams, auglin_unigrams),
    ("bigrams", shap_bigrams, auglin_bigrams),
    ("trigrams", shap_trigrams, auglin_trigrams),
]:
    n_overlap, shared = overlap(shap_top, auglin_top)
    print(f"{n_words}: {n_overlap}/{min(len(shap_top), len(auglin_top))} shared in both top-15 -> {shared}")

unigrams: 1/15 shared in both top-15 -> ['cardiology']
bigrams: 4/15 shared in both top-15 -> ['chest pain', 'heart disease', 'hypertensive heart', 'valvular heart']
trigrams: 1/15 shared in both top-15 -> ['hypertensive heart disease']


## 7. Performance: how much does interpretability cost?

BERT (full fine-tune, black box) vs. Aug-Linear (interpretable by construction) vs.
TF-IDF (block 3's simplest baseline, for reference) -- same held-out test fold, same
weak-label training target, all evaluated against both the weak and the real cardiac
label.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))
X_fit_tfidf = tfidf_vectorizer.fit_transform(texts[fit_idx_sub])
tfidf_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
tfidf_clf.fit(X_fit_tfidf, weak_y[fit_idx_sub])
tfidf_test_proba = tfidf_clf.predict_proba(tfidf_vectorizer.transform(texts[test_idx]))[:, 1]
tfidf_val_proba = tfidf_clf.predict_proba(tfidf_vectorizer.transform(texts[val_idx]))[:, 1]

bert_val_proba = bert_predict_proba(bert_model, texts[val_idx])
auglin_val_proba = auglin_clf.predict_proba(doc_embeddings[val_idx])[:, 1]

def compute_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }

model_runs = [
    ("BERT (full fine-tune)", test_proba, bert_val_proba),
    ("Aug-Linear (BERT n-gram embeddings)", auglin_test_proba, auglin_val_proba),
    ("TF-IDF + LogisticRegression", tfidf_test_proba, tfidf_val_proba),
]

rows = []
for name, test_proba_m, val_proba_m in model_runs:
    tuned_threshold = pick_threshold(weak_y[val_idx], val_proba_m)
    for label_type, y in [("weak", weak_y[test_idx]), ("real", real_y[test_idx])]:
        for threshold_name, threshold in [("fixed_0.5", 0.5), ("tuned", tuned_threshold)]:
            m = compute_metrics(y, test_proba_m, threshold)
            rows.append({"model": name, "evaluated_against": label_type, "threshold": threshold_name, **m})

results = pd.DataFrame(rows)
results = results.sort_values(["model", "evaluated_against", "threshold"]).reset_index(drop=True)
results.round(3)

,model,evaluated_against,threshold,precision,recall,f1,roc_auc,pr_auc
0,Aug-Linear (BERT n-gram embeddings),real,fixed_0.5,1.000,0.006,0.012,0.822,0.717
1,Aug-Linear (BERT n-gram embeddings),real,tuned,0.907,0.242,0.382,0.822,0.717
2,Aug-Linear (BERT n-gram embeddings),weak,fixed_0.5,1.000,0.022,0.043,0.792,0.335
3,Aug-Linear (BERT n-gram embeddings),weak,tuned,0.302,0.289,0.295,0.792,0.335
4,BERT (full fine-tune),real,fixed_0.5,0.959,0.292,0.448,0.859,0.776
5,BERT (full fine-tune),real,tuned,0.880,0.410,0.559,0.859,0.776
6,BERT (full fine-tune),weak,fixed_0.5,0.306,0.333,0.319,0.831,0.304
7,BERT (full fine-tune),weak,tuned,0.293,0.489,0.367,0.831,0.304
8,TF-IDF + LogisticRegression,real,fixed_0.5,1.000,0.050,0.095,0.839,0.754
9,TF-IDF + LogisticRegression,real,tuned,0.787,0.528,0.632,0.839,0.754


## 8. Discussion

**Both methods surface clinically coherent terms, once SHAP is fixed to operate on
whole words.** The first pass at this notebook masked and reported BERT's own WordPiece
sub-tokens ("card" + "iology", "pro" + "ximal"), which produced unreadable fragments --
switching the masker to split on whitespace/punctuation instead fixed that, and SHAP's
top terms are now legible: `ef`, `af`, `lad`, `tte` (standard cardiology abbreviations),
`chest pain`, `hypertensive heart disease`.

**Aug-Linear's original, much weaker performance turned out to be a fixable
implementation bug, not a limitation of the method.** The first version represented each
document as the raw *sum* of its n-grams' BERT embeddings, with no length
normalization -- unlike TF-IDF, whose vectors are L2-normalized by default, so a longer,
more verbose note doesn't automatically get a larger-magnitude feature vector regardless
of its actual cardiac-ness. Testing that hypothesis directly: L2-normalizing the summed
embeddings took Aug-Linear from clearly the weakest model (ROC-AUC 0.644, PR-AUC 0.537)
to nearly matching TF-IDF and BERT (ROC-AUC 0.822, PR-AUC 0.717, vs. 0.839/0.754 and
0.859/0.776) -- confirming that most of the earlier gap was the length confound, not a
fundamental weakness of summing BERT n-gram embeddings. Mean-pooling instead of summing
also helped (0.788/0.681) but less than proper L2 normalization. The corrected top
n-grams are, if anything, even more specific and clinically sharp: `myocardial`,
`cardiomyopathy`, `echocardiogram`, `clopidogrel`, `atorvastatin` as unigrams;
`echocardiogram ef`, `hypokinetic cardiomyopathy`, `clopidogrel 75 mg` as bigrams/
trigrams -- real cardiac medications and diagnostic concepts, not documentation
boilerplate.

**One threshold-related wrinkle survived the fix**: F1 against the real label actually
*dropped* slightly after normalizing (0.412 -> 0.382) even though ROC-AUC and PR-AUC
improved substantially. This is the same pitfall from block 3 in a new guise -- the
decision threshold is tuned to maximise F1 against the *weak* label (the only
supervision available without leakage), and a better-ranking model doesn't automatically
get a threshold that happens to transfer well to the real label at that specific cutoff.
The threshold-free metrics are the trustworthy comparison here, and by those, the fix
clearly worked.

**SHAP and Aug-Linear's top-15 lists now overlap a little, where they used to overlap
not at all**: 1/15 shared unigrams (`cardiology`), 4/15 bigrams (`chest pain`, `heart
disease`, `hypertensive heart`, `valvular heart`), 1/15 trigrams (`hypertensive heart
disease`). Removing the length confound let Aug-Linear's ranking reflect genuine
per-n-gram signal more cleanly, which converges partway with what SHAP finds important
in the actual fine-tuned model -- but the overlap is still far from complete, because
they remain genuinely different objects: SHAP explains *this specific BERT model's*
local behaviour on 20 sampled notes; Aug-Linear reports a *different, linear* model's
global coefficients fit on ~675 notes. Fixing an implementation bug closes some of the
gap between them, not all of it -- agreement was never guaranteed just because both
start from BERT.

**Practical takeaway, revised**: SHAP is still the right tool for explaining one
specific, already-deployed black-box prediction. Aug-Linear, once implemented
correctly (embeddings summed *and* length-normalized), is a much more credible
alternative than the first pass suggested -- a genuinely competitive, inherently
interpretable model, not just a cheaper-but-worse one. The practical lesson worth
carrying forward: an "interpretable model performs worse" result is worth a second look
before accepting it at face value -- here, a missing normalization step, not the
underlying idea, was responsible for most of the gap.